In [ ]:
# Install required packages (runs automatically in Colab, fast no-op in Binder)
!pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-ibm-catalog qiskit-serverless

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# SABREによるトランスパイル最適化
*使用時間の目安: Heron r2プロセッサで1分未満（注意: これはあくまで目安です。実際の実行時間は異なる場合があります。）*
## 背景
トランスパイルはQiskitにおける重要なステップであり、量子回路を特定の量子ハードウェアと互換性のある形式に変換します。これには2つの主要な段階が含まれます: **量子ビットレイアウト**（論理量子ビットをデバイス上の物理量子ビットにマッピングすること）と**ゲートルーティング**（必要に応じてSWAPゲートを挿入し、マルチ量子ビットゲートがデバイスの接続性を満たすようにすること）です。

SABRE（*SWAP-Based Bidirectional heuristic search algorithm*）は、レイアウトとルーティングの両方に対応する強力な最適化ツールです。**大規模回路**（100量子ビット以上）や、**IBM&reg; Heron**のような複雑なカップリングマップを持つデバイスに対して特に効果的であり、量子ビットマッピングの指数関数的な増大に対して効率的な解決策を提供します。

### SABREを使用する理由
SABREはSWAPゲートの数を最小化し、回路の深さを削減することで、実際のハードウェア上での回路性能を向上させます。そのヒューリスティックベースのアプローチは、高度なハードウェアや大規模で複雑な回路に最適です。[LightSABRE](https://arxiv.org/abs/2409.08368)アルゴリズムで導入された最近の改良により、SABREの性能がさらに最適化され、より高速な実行時間とより少ないSWAPゲート数が実現されています。これらの改善により、大規模回路に対する効果がさらに高まっています。

### 学習内容
このチュートリアルは2つのパートに分かれています:
1. 大規模回路の高度な最適化のために、**Qiskitパターン**でSABREを使用する方法を学びます。
2. **qiskit_serverless**を活用して、スケーラブルで効率的なトランスパイルにおけるSABREの潜在能力を最大限に引き出します。

具体的には以下の内容を扱います:
- `optimization_level=3`などのデフォルトのトランスパイル設定を超えて、100量子ビット以上の回路に対してSABREを最適化します。
- 実行時間を改善しゲート数を削減する**LightSABREの改良点**を探ります。
- SABREの主要パラメータ（`swap_trials`、`layout_trials`、`max_iterations`、`heuristic`）をカスタマイズして、**回路品質**と**トランスパイル実行時間**のバランスを取ります。
## 前提条件
このチュートリアルを始める前に、以下がインストールされていることを確認してください:
- Qiskit SDK v1.0以降、[visualization](https://docs.quantum.ibm.com/api/qiskit/visualization)サポート付き
- Qiskit Runtime v0.28以降（`pip install qiskit-ibm-runtime`）
- Serverless（`pip install qiskit-ibm-catalog qiskit_serverless`）
## セットアップ

In [1]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit_ibm_catalog import QiskitServerless, QiskitFunction
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorOptions
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit.transpiler import CouplingMap
from qiskit.transpiler.passes import SabreLayout, SabreSwap
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
import matplotlib.pyplot as plt
import numpy as np
import time

## パートI. QiskitパターンでのSABREの使用

SABREはQiskitで量子回路を最適化するために使用でき、量子ビットレイアウトとゲートルーティングの両方の段階を処理します。このセクションでは、QiskitパターンでSABREを使用する**最小限の例**を紹介します。主な焦点はステップ2の最適化にあります。

SABREを実行するには、以下が必要です:
- 量子回路の**DAG**（有向非巡回グラフ）表現。
- バックエンドからの**カップリングマップ**。量子ビットが物理的にどのように接続されているかを指定します。
- **SABREパス**。レイアウトとルーティングを最適化するアルゴリズムを適用します。

このパートでは、**SabreLayout**パスに焦点を当てます。SabreLayoutはレイアウトとルーティングの両方の試行を実行し、必要なSWAPゲート数を最小化しながら最も効率的な初期レイアウトを見つけます。重要な点として、`SabreLayout`は単独で、最も少ないSWAPゲート数を追加する解を内部的にレイアウトとルーティングの両方で最適化します。**SabreLayout**のみを使用する場合、SABREのヒューリスティックを変更することはできませんが、`layout_trials`の数はカスタマイズ可能です。

### ステップ1: 古典的な入力を量子問題にマッピングする

**GHZ（Greenberger-Horne-Zeilinger）**回路は、すべての量子ビットが`|0...0⟩`または`|1...1⟩`状態のいずれかにあるエンタングル状態を準備する量子回路です。$n$量子ビットのGHZ状態は数学的に以下のように表されます:
$$ |\text{GHZ}\rangle = \frac{1}{\sqrt{2}} \left( |0\rangle^{\otimes n} + |1\rangle^{\otimes n} \right) $$

以下の操作を適用して構成されます:
1. 最初の量子ビットにアダマールゲートを適用して重ね合わせを作成します。
2. 残りの量子ビットを最初の量子ビットとエンタングルするために一連のCNOTゲートを適用します。

この例では、線形トポロジではなく、意図的に**スター型トポロジのGHZ回路**を構成します。スター型トポロジでは、最初の量子ビットが「ハブ」として機能し、他のすべての量子ビットがCNOTゲートを使用して直接エンタングルされます。この選択は意図的なものです。**線形トポロジのGHZ状態**は理論的にはSWAPゲートなしで線形カップリングマップ上で$ O(N) $の深さで実装できますが、SABREは100量子ビットのGHZ回路をバックエンドのheavy-hexカップリングマップの部分グラフにマッピングすることで自明に最適解を見つけてしまうためです。

**スター型トポロジのGHZ回路**は、はるかに困難な問題を提示します。理論的にはSWAPゲートなしで$ O(N) $の深さで実行可能ですが、この解を見つけるには最適な初期レイアウトを特定する必要があり、回路の非線形な接続性のためにこれは非常に困難です。このトポロジは、SABREの評価においてより優れたテストケースとなり、設定パラメータがより複雑な条件下でレイアウトとルーティングの性能にどのように影響するかを示します。

![ghz_star_topology.png](../docs/images/tutorials/transpilation-optimizations-with-sabre/ghz_star_topology.avif)

注目すべき点:
- **HighLevelSynthesis**ツールは、上の画像に示されているように、SWAPゲートを導入せずにスター型トポロジGHZ回路の最適な$ O(N) $深さの解を生成できます。
- あるいは、**StarPrerouting**パスはSABREのルーティング決定を導くことで深さをさらに削減できますが、一部のSWAPゲートが導入される可能性があります。ただし、StarPreroutingは実行時間を増加させ、初期トランスパイルプロセスへの統合が必要です。

このチュートリアルでは、SABREの設定が実行時間と回路の深さに与える直接的な影響を分離して強調するため、HighLevelSynthesisとStarPreroutingの両方を除外します。各量子ビットペアの期待値$ \langle Z_0 Z_i \rangle $を測定することで、以下を分析します:
- SABREがSWAPゲートと回路の深さをどの程度削減するか。
- これらの最適化が実行された回路の忠実度に与える影響。$ \langle Z_0 Z_i \rangle = 1 $からの逸脱はエンタングルメントの損失を示します。

In [2]:
# set seed for reproducibility
seed = 42
num_qubits = 110

# Create GHZ circuit
qc = QuantumCircuit(num_qubits)
qc.h(0)
for i in range(1, num_qubits):
    qc.cx(0, i)

qc.measure_all()

次に、システムの挙動を評価するために対象の演算子をマッピングします。具体的には、量子ビット間の`ZZ`演算子を使用して、量子ビットが離れるにつれてエンタングルメントがどのように劣化するかを調べます。この分析は重要です。離れた量子ビットに対する期待値$\langle Z_0 Z_i \rangle$の不正確さが、回路実行におけるノイズやエラーの影響を明らかにするためです。これらの逸脱を研究することで、異なるSABRE設定の下で回路がエンタングルメントをどの程度保持するか、またSABREがハードウェア制約の影響をどの程度効果的に最小化するかについての知見が得られます。

In [3]:
# ZZII...II, ZIZI...II, ... , ZIII...IZ
operator_strings = [
    "Z" + "I" * i + "Z" + "I" * (num_qubits - 2 - i)
    for i in range(num_qubits - 1)
]
print(operator_strings)
print(len(operator_strings))

operators = [SparsePauliOp(operator) for operator in operator_strings]

['ZZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII

### Step 2: Optimize problem for quantum hardware execution

In this step, we focus on optimizing the circuit layout for execution on a specific quantum hardware device with 127 qubits. This is the main focus of the tutorial, as we perform **SABRE optimizations and transpilation** to achieve the best circuit performance. Using the `SabreLayout` pass, we determine an initial qubit mapping that minimizes the need for SWAP gates during routing. By passing the `coupling_map` of the target backend, `SabreLayout` adapts the layout to the device's connectivity constraints.

We will use `generate_preset_pass_manager` with `optimization_level=3` for the transpilation process and customize the `SabreLayout` pass with different configurations. The goal is to find a setup that produces a transpiled circuit with the **lowest size and/or depth**, demonstrating the impact of SABRE optimizations.

#### Why Are Circuit Size and Depth Important?

- **Lower size (gate count):** Reduces the number of operations, minimizing opportunities for errors to accumulate.
- **Lower depth:** Shortens the overall execution time, which is critical for avoiding decoherence and maintaining quantum state fidelity.

By optimizing these metrics, we improve the circuit’s reliability and execution accuracy on noisy quantum hardware.

Select the backend.

In [4]:
service = QiskitRuntimeService()
# backend = service.least_busy(
#    operational=True, simulator=False, min_num_qubits=127
# )
backend = service.backend("ibm_boston")
print(f"Using backend: {backend.name}")

Using backend: ibm_boston


### ステップ2: 量子ハードウェア実行のための問題の最適化
このステップでは、127量子ビットの特定の量子ハードウェアデバイスでの実行に向けて回路レイアウトの最適化に焦点を当てます。これがチュートリアルの主要な焦点であり、最高の回路性能を達成するために**SABREの最適化とトランスパイル**を実行します。`SabreLayout`パスを使用して、ルーティング中に必要なSWAPゲートを最小化する初期量子ビットマッピングを決定します。ターゲットバックエンドの`coupling_map`を渡すことで、`SabreLayout`はデバイスの接続性制約にレイアウトを適応させます。

トランスパイルプロセスには`optimization_level=3`の`generate_preset_pass_manager`を使用し、異なる設定で`SabreLayout`パスをカスタマイズします。目標は、**最小のサイズおよび/または深さ**を持つトランスパイル済み回路を生成する設定を見つけ、SABREの最適化の効果を示すことです。

#### 回路のサイズと深さが重要な理由
- **サイズ（ゲート数）が小さい:** 操作の数を減らし、エラーが蓄積する機会を最小化します。
- **深さが浅い:** 全体的な実行時間を短縮し、デコヒーレンスの回避と量子状態の忠実度の維持に重要です。

これらのメトリクスを最適化することで、ノイズのある量子ハードウェア上での回路の信頼性と実行精度を向上させます。
バックエンドを選択します。

In [5]:
# Get the coupling map from the backend
cmap = CouplingMap(backend().configuration().coupling_map)

# Create the SabreLayout passes for the custom configurations
sl_2 = SabreLayout(
    coupling_map=cmap,
    seed=seed,
    max_iterations=4,
    layout_trials=200,
    swap_trials=200,
)
sl_3 = SabreLayout(
    coupling_map=cmap,
    seed=seed,
    max_iterations=8,
    layout_trials=200,
    swap_trials=200,
)

# Create the pass managers, need to first create then configure the SabreLayout passes
pm_1 = generate_preset_pass_manager(
    optimization_level=3, backend=backend, seed_transpiler=seed
)
pm_2 = generate_preset_pass_manager(
    optimization_level=3, backend=backend, seed_transpiler=seed
)
pm_3 = generate_preset_pass_manager(
    optimization_level=3, backend=backend, seed_transpiler=seed
)

Now we can configure the `SabreLayout` pass in the custom pass managers. To do this we know that for the default `generate_preset_pass_manager` on `optimization_level=3`, the `SabreLayout` pass is at index 2, as `SabreLayout` occurs after `SetLayout` and `VF2Laout` passes. We can access this pass and modify its parameters.

In [6]:
pm_2.layout.replace(index=2, passes=sl_2)
pm_3.layout.replace(index=2, passes=sl_3)

回路最適化に対する異なる設定の影響を評価するために、`SabreLayout`パスの固有の設定を持つ3つのパスマネージャーを作成します。これらの設定は、回路品質とトランスパイル時間のトレードオフを分析するのに役立ちます。

#### 主要パラメータ
- **`max_iterations`**: レイアウトを改良しルーティングコストを削減するための順方向・逆方向ルーティング反復の回数です。
- **`layout_trials`**: テストされるランダムな初期レイアウトの数で、SWAPゲートを最小化するものが選択されます。
- **`swap_trials`**: 各レイアウトに対するルーティング試行の回数で、より良いルーティングのためにゲート配置を改良します。

`layout_trials`と`swap_trials`を増やすと、より徹底的な最適化が行われますが、トランスパイル時間が増加します。

#### このチュートリアルでの設定
1. **`pm_1`**: `optimization_level=3`のデフォルト設定です。
   - `max_iterations=4`
   - `layout_trials=20`
   - `swap_trials=20`

2. **`pm_2`**: より良い探索のために試行回数を増やします。
   - `max_iterations=4`
   - `layout_trials=200`
   - `swap_trials=200`

3. **`pm_3`**: `pm_2`を拡張し、さらなる改良のために反復回数を増やします。
   - `max_iterations=8`
   - `layout_trials=200`
   - `swap_trials=200`

これらの設定の結果を比較することで、回路品質（例: サイズと深さ）と計算コストの最適なバランスを達成する設定を特定することを目指します。

In [7]:
# Transpile the circuit with each pass manager and measure the time
t0 = time.time()
tqc_1 = pm_1.run(qc)
t1 = time.time() - t0
t0 = time.time()
tqc_2 = pm_2.run(qc)
t2 = time.time() - t0
t0 = time.time()
tqc_3 = pm_3.run(qc)
t3 = time.time() - t0

# Obtain the depths and the total number of gates (circuit size)
depth_1 = tqc_1.depth(lambda x: x.operation.num_qubits == 2)
depth_2 = tqc_2.depth(lambda x: x.operation.num_qubits == 2)
depth_3 = tqc_3.depth(lambda x: x.operation.num_qubits == 2)
size_1 = tqc_1.size()
size_2 = tqc_2.size()
size_3 = tqc_3.size()

# Transform the observables to match the backend's ISA
operators_list_1 = [op.apply_layout(tqc_1.layout) for op in operators]
operators_list_2 = [op.apply_layout(tqc_2.layout) for op in operators]
operators_list_3 = [op.apply_layout(tqc_3.layout) for op in operators]

# Compute improvements compared to pass manager 1 (default)
depth_improvement_2 = ((depth_1 - depth_2) / depth_1) * 100
depth_improvement_3 = ((depth_1 - depth_3) / depth_1) * 100
size_improvement_2 = ((size_1 - size_2) / size_1) * 100
size_improvement_3 = ((size_1 - size_3) / size_1) * 100
time_increase_2 = ((t2 - t1) / t1) * 100
time_increase_3 = ((t3 - t1) / t1) * 100

print(
    f"Pass manager 1 (4,20,20)  : Depth {depth_1}, Size {size_1}, Time {t1:.4f} s"
)
print(
    f"Pass manager 2 (4,200,200): Depth {depth_2}, Size {size_2}, Time {t2:.4f} s"
)
print(f"  - Depth improvement: {depth_improvement_2:.2f}%")
print(f"  - Size improvement: {size_improvement_2:.2f}%")
print(f"  - Time increase: {time_increase_2:.2f}%")
print(
    f"Pass manager 3 (8,200,200): Depth {depth_3}, Size {size_3}, Time {t3:.4f} s"
)
print(f"  - Depth improvement: {depth_improvement_3:.2f}%")
print(f"  - Size improvement: {size_improvement_3:.2f}%")
print(f"  - Time increase: {time_increase_3:.2f}%")

Pass manager 1 (4,20,20)  : Depth 439, Size 2346, Time 0.5775 s
Pass manager 2 (4,200,200): Depth 395, Size 2070, Time 3.9927 s
  - Depth improvement: 10.02%
  - Size improvement: 11.76%
  - Time increase: 591.43%
Pass manager 3 (8,200,200): Depth 375, Size 1873, Time 2.3079 s
  - Depth improvement: 14.58%
  - Size improvement: 20.16%
  - Time increase: 299.67%


次に、カスタムパスマネージャーで`SabreLayout`パスを設定します。`optimization_level=3`のデフォルトの`generate_preset_pass_manager`では、`SabreLayout`パスはインデックス2にあります。これは`SabreLayout`が`SetLayout`と`VF2Laout`パスの後に発生するためです。このパスにアクセスしてパラメータを変更できます。

In [8]:
# Plot the results of the metrics
times = [t1, t2, t3]
depths = [depth_1, depth_2, depth_3]
sizes = [size_1, size_2, size_3]
pm_names = [
    "pm_1 (4 iter, 20 trials)",
    "pm_2 (4 iter, 200 trials)",
    "pm_3 (8 iter, 200 trials)",
]
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(pm_names)))

# Create a figure with three subplots
fig, axs = plt.subplots(3, 1, figsize=(6, 9), sharex=True)
axs[0].bar(pm_names, times, color=colors)
axs[0].set_ylabel("Time (s)", fontsize=12)
axs[0].set_title("Transpilation Time", fontsize=14)
axs[0].grid(axis="y", linestyle="--", alpha=0.7)
axs[1].bar(pm_names, depths, color=colors)
axs[1].set_ylabel("Depth", fontsize=12)
axs[1].set_title("Circuit Depth", fontsize=14)
axs[1].grid(axis="y", linestyle="--", alpha=0.7)
axs[2].bar(pm_names, sizes, color=colors)
axs[2].set_ylabel("Size", fontsize=12)
axs[2].set_title("Circuit Size", fontsize=14)
axs[2].set_xticks(range(len(pm_names)))
axs[2].set_xticklabels(pm_names, fontsize=10, rotation=15)
axs[2].grid(axis="y", linestyle="--", alpha=0.7)

# Add some spacing between subplots
plt.tight_layout()
plt.show()

<Image src="../docs/images/tutorials/transpilation-optimizations-with-sabre/extracted-outputs/818a8997-d2c7-4661-a6ea-f58eac376bf8-0.avif" alt="Output of the previous code cell" />

各パスマネージャーの設定が完了したので、それぞれのトランスパイルプロセスを実行します。結果を比較するために、トランスパイル時間、回路の深さ（2量子ビットゲートの深さとして測定）、およびトランスパイル済み回路の合計ゲート数を含む主要なメトリクスを追跡します。

In [9]:
options = EstimatorOptions()
options.resilience_level = 2
options.dynamical_decoupling.enable = True
options.dynamical_decoupling.sequence_type = "XY4"

# Create an Estimator object
estimator = Estimator(backend, options=options)

In [10]:
# Submit the circuit to Estimator
job_1 = estimator.run([(tqc_1, operators_list_1)])
job_1_id = job_1.job_id()
print(job_1_id)

job_2 = estimator.run([(tqc_2, operators_list_2)])
job_2_id = job_2.job_id()
print(job_2_id)

job_3 = estimator.run([(tqc_3, operators_list_3)])
job_3_id = job_3.job_id()
print(job_3_id)

d5k0qs7853es738dab6g
d5k0qsf853es738dab70
d5k0qsf853es738dab7g


In [11]:
# Run the jobs
result_1 = job_1.result()[0]
print("Job 1 done")
result_2 = job_2.result()[0]
print("Job 2 done")
result_3 = job_3.result()[0]
print("Job 3 done")

Job 1 done
Job 2 done
Job 3 done


![Output of the previous code cell](../docs/images/tutorials/transpilation-optimizations-with-sabre/extracted-outputs/818a8997-d2c7-4661-a6ea-f58eac376bf8-0.avif)

### ステップ3: Qiskitプリミティブを使用して実行する
このステップでは、`Estimator`プリミティブを使用して`ZZ`演算子の期待値$\langle Z_0 Z_i \rangle$を計算し、トランスパイル済み回路のエンタングルメントと実行品質を評価します。一般的なユーザーワークフローに合わせて、ジョブを実行に送信し、**動的デカップリング**を使用してエラー抑制を適用します。動的デカップリングは、量子ビット状態を保持するためにゲートシーケンスを挿入してデコヒーレンスを軽減する技術です。さらに、ノイズに対抗するためにレジリエンスレベルを指定します。レベルが高いほどより正確な結果が得られますが、処理時間が増加します。このアプローチにより、現実的な実行条件下での各パスマネージャー設定の性能を評価します。

In [12]:
data = list(range(1, len(operators) + 1))  # Distance between the Z operators

values_1 = list(result_1.data.evs)
values_2 = list(result_2.data.evs)
values_3 = list(result_3.data.evs)

plt.plot(
    data,
    values_1,
    marker="o",
    label="pm_1 (iters=4, swap_trials=20, layout_trials=20)",
)
plt.plot(
    data,
    values_2,
    marker="s",
    label="pm_2 (iters=4, swap_trials=200, layout_trials=200)",
)
plt.plot(
    data,
    values_3,
    marker="^",
    label="pm_3 (iters=8, swap_trials=200, layout_trials=200)",
)
plt.xlabel("Distance between qubits $i$")
plt.ylabel(r"$\langle Z_i Z_0 \rangle / \langle Z_1 Z_0 \rangle $")
plt.legend()
plt.show()

<Image src="../docs/images/tutorials/transpilation-optimizations-with-sabre/extracted-outputs/bc6cb36f-4bf2-4275-baf5-9557fcba520a-0.avif" alt="Output of the previous code cell" />

### Analysis of Results

The plot shows the expectation values $\langle Z_0 Z_i \rangle / \langle Z_0 Z_0 \rangle$  as a function of the distance between qubits for three pass manager configurations with increasing levels of optimization. In the ideal case, these values remain close to 1, indicating strong correlations across the circuit. As the distance increases, noise and accumulated errors lead to a decay in correlations, revealing how well each transpilation strategy preserves the underlying structure of the state.

Among the three configurations, `pm_1` clearly performs the worst. Its correlation values decay rapidly as the distance increases and approach zero much earlier than the other two configurations. This behavior is consistent with its larger circuit depth and gate count, where accumulated noise quickly degrades long-range correlations.

Both `pm_2` and `pm_3` represent significant improvements over `pm_1` across essentially all distances. On average, `pm_3` exhibits the strongest overall performance, maintaining higher correlation values over longer distances and showing a more gradual decay. This aligns with its more aggressive optimization, which produces shallower circuits that are generally more robust to noise accumulation.

That said, `pm_2` shows noticeably better accuracy at short distances compared to `pm_3`, despite having a slightly larger depth and gate count. This suggests that circuit depth alone does not fully determine performance; the specific structure produced by the transpilation, including how entangling gates are arranged and how errors propagate through the circuit, also plays an important role. In some cases, the transformations applied by `pm_2` appear to better preserve local correlations, even if they do not scale as well to longer distances.

Taken together, these results highlight a trade-off between circuit compactness and circuit structure. While increased optimization generally improves long-range stability, the best performance for a given observable depends on both reducing circuit depth and producing a structure that is well matched to the noise characteristics of the hardware.

## Part II. Configuring the heuristic in SABRE and using Serverless

In addition to adjusting trial numbers, SABRE supports customization of the routing heuristic used during transpilation. By default, `SabreLayout` employs the decay heuristic, which dynamically weights qubits based on their likelihood of being swapped. To use a different heuristic (such as the `lookahead` heuristic), you can create a custom `SabreSwap` pass and connect it to `SabreLayout` by running a `PassManager` with `FullAncillaAllocation`, `EnlargeWithAncilla`, and `ApplyLayout`. When using `SabreSwap` as a parameter for `SabreLayout`, only one layout trial is performed by default. To efficiently run multiple layout trials, we leverage the serverless runtime for parallelization. For more about serverless, see the [Serverless documentation](/docs/guides/serverless).

### How to Change the Routing Heuristic
1. Create a custom `SabreSwap` pass with the desired heuristic.
2. Use this custom `SabreSwap` as the routing method for the `SabreLayout` pass.

While it is possible to run multiple layout trials using a loop, serverless runtime is the better choice for large-scale and more vigorous experiments. Serverless supports parallel execution of layout trials, significantly speeding up the optimization of larger circuits and large experimental sweeps. This makes it especially valuable when working with resource-intensive tasks or when time efficiency is critical.

This section focuses solely on step 2 of optimization: minimizing circuit size and depth to achieve the best possible transpiled circuit. Building on the earlier results, we now explore how heuristic customization and serverless parallelization can further enhance optimization performance, making it suitable for large-scale quantum circuit transpilation.

### Results without serverless runtime (1 layout trial):

In [17]:
swap_trials = 1000

# Default PassManager with `SabreLayout` and `SabreSwap`, using heuristic "decay"
sr_default = SabreSwap(
    coupling_map=cmap, heuristic="decay", trials=swap_trials, seed=seed
)
sl_default = SabreLayout(
    coupling_map=cmap, routing_pass=sr_default, seed=seed
)
pm_default = generate_preset_pass_manager(
    optimization_level=3, backend=backend, seed_transpiler=seed
)
pm_default.layout.replace(index=2, passes=sl_default)
pm_default.routing.replace(index=1, passes=sr_default)

t0 = time.time()
tqc_default = pm_default.run(qc)
t_default = time.time() - t0
size_default = tqc_default.size()
depth_default = tqc_default.depth(lambda x: x.operation.num_qubits == 2)


# Custom PassManager with `SabreLayout` and `SabreSwap`, using heuristic "lookahead"
sr_custom = SabreSwap(
    coupling_map=cmap, heuristic="lookahead", trials=swap_trials, seed=seed
)
sl_custom = SabreLayout(coupling_map=cmap, routing_pass=sr_custom, seed=seed)
pm_custom = generate_preset_pass_manager(
    optimization_level=3, backend=backend, seed_transpiler=seed
)
pm_custom.layout.replace(index=2, passes=sl_custom)
pm_custom.routing.replace(index=1, passes=sr_custom)

t0 = time.time()
tqc_custom = pm_custom.run(qc)
t_custom = time.time() - t0
size_custom = tqc_custom.size()
depth_custom = tqc_custom.depth(lambda x: x.operation.num_qubits == 2)

print(
    f"Default (heuristic='decay')    : Depth {depth_default}, Size {size_default}, Time {t_default}"
)
print(
    f"Custom  (heuristic='lookahead'): Depth {depth_custom}, Size {size_custom}, Time {t_custom}"
)

Default (heuristic='decay')    : Depth 443, Size 3115, Time 1.034372091293335
Custom  (heuristic='lookahead'): Depth 432, Size 2856, Time 0.6669301986694336


Here we see that the `lookahead` heuristic performs better than the `decay` heuristic in terms of circuit depth, size, and time. This improvements highlights how we can improve SABRE beyond just trials and iterations for your specific circuit and hardware constraints. Note that these results are based on a single layout trial. To achieve more accurate results, we recommend running multiple layout trials, which can be done efficiently using the serverless runtime.

### Results with serverless runtime (multiple layout trials)

Qiskit Serverless requires setting up your workload’s `.py` files into a dedicated directory. The following code cell is a Python file in the `source_files` directory named `transpile_remote.py`. This file contains the function that runs the transpilation process.

In [18]:
# This cell is hidden from users, it makes sure the `source_files` directory exists
from pathlib import Path

Path("source_files").mkdir(exist_ok=True)

In [26]:
%%writefile source_files/transpile_remote.py
import time
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.transpiler.passes import SabreLayout, SabreSwap
from qiskit.transpiler import CouplingMap
from qiskit_serverless import get_arguments, save_result, distribute_task, get
from qiskit_ibm_runtime import QiskitRuntimeService

@distribute_task(target={
    "cpu": 1,
    "mem": 1024 * 1024 * 1024
})
def transpile_remote(qc, optimization_level, backend_name, seed, swap_trials, heuristic):
    """Transpiles an abstract circuit into an ISA circuit for a given backend."""

    service = QiskitRuntimeService()
    backend = service.backend(backend_name)

    pm = generate_preset_pass_manager(
        optimization_level=optimization_level,
        backend=backend,
        seed_transpiler=seed
    )

    # Changing the `SabreLayout` and `SabreSwap` passes to use the custom configurations
    cmap = CouplingMap(backend().configuration().coupling_map)
    sr = SabreSwap(coupling_map=cmap, heuristic=heuristic, trials=swap_trials, seed=seed)
    sl = SabreLayout(coupling_map=cmap, routing_pass=sr, seed=seed)
    pm.layout.replace(index=2, passes=sl)
    pm.routing.replace(index=1, passes=sr)

    # Measure the transpile time
    start_time = time.time()  # Start timer
    tqc = pm.run(qc)  # Transpile the circuit
    end_time = time.time()  # End timer

    transpile_time = end_time - start_time  # Calculate the elapsed time
    return tqc, transpile_time  # Return both the transpiled circuit and the transpile time


# Get program arguments
arguments = get_arguments()
circuit = arguments.get("circuit")
backend_name = arguments.get("backend_name")
optimization_level = arguments.get("optimization_level")
seed_list = arguments.get("seed_list")
swap_trials = arguments.get("swap_trials")
heuristic = arguments.get("heuristic")

# Transpile the circuits
transpile_worker_references = [
    transpile_remote(circuit, optimization_level, backend_name, seed, swap_trials, heuristic)
    for seed in seed_list
]

results_with_times = get(transpile_worker_references)

# Separate the transpiled circuits and their transpile times
transpiled_circuits = [result[0] for result in results_with_times]
transpile_times = [result[1] for result in results_with_times]

# Save both results and transpile times
save_result({"transpiled_circuits": transpiled_circuits, "transpile_times": transpile_times})

Overwriting source_files/transpile_remote.py


The following cell uploads the `transpile_remote.py` file as a Qiskit Serverless program under the name `transpile_remote_serverless`.

In [27]:
serverless = QiskitServerless()

transpile_remote_demo = QiskitFunction(
    title="transpile_remote_serverless",
    entrypoint="transpile_remote.py",
    working_dir="./source_files/",
)
serverless.upload(transpile_remote_demo)
transpile_remote_serverless = serverless.load("transpile_remote_serverless")

### ステップ4: 後処理を行い、望ましい古典形式で結果を返す
ジョブが完了したら、各量子ビットの期待値$\langle Z_0 Z_i \rangle$をプロットして結果を分析します。理想的なシミュレーションでは、すべての$\langle Z_0 Z_i \rangle$値は1であり、量子ビット全体にわたる完全なエンタングルメントを反映します。しかし、ノイズやハードウェアの制約により、`i`が増加するにつれて期待値は通常減少し、距離に応じてエンタングルメントがどのように劣化するかが明らかになります。

このステップでは、各パスマネージャー設定の結果を理想的なシミュレーションと比較します。各設定における$\langle Z_0 Z_i \rangle$の1からの逸脱を調べることで、各パスマネージャーがエンタングルメントをどの程度保持し、ノイズの影響をどの程度軽減するかを定量化できます。この分析は、SABREの最適化が実行の忠実度に与える影響を直接評価し、最適化品質と実行性能のバランスが最も優れた設定を明らかにします。

結果はパスマネージャー間の差異を強調するように可視化され、レイアウトとルーティングの改善がノイズのある量子ハードウェア上での最終的な回路実行にどのように影響するかを示します。

In [28]:
num_seeds = 20  # represents the different layout trials
seed_list = [seed + i for i in range(num_seeds)]

![Output of the previous code cell](../docs/images/tutorials/transpilation-optimizations-with-sabre/extracted-outputs/bc6cb36f-4bf2-4275-baf5-9557fcba520a-0.avif)

### 結果の分析
プロットは、最適化レベルが異なる3つのパスマネージャー設定について、量子ビット間の距離の関数として期待値$\langle Z_0 Z_i \rangle / \langle Z_0 Z_0 \rangle$を示しています。理想的な場合、これらの値は1に近い状態を維持し、回路全体にわたる強い相関を示します。距離が増加するにつれて、ノイズと蓄積されたエラーが相関の減衰を引き起こし、各トランスパイル戦略が状態の基本構造をどの程度保持しているかが明らかになります。

3つの設定の中で、`pm_1`は明らかに最も性能が劣っています。距離が増加するにつれて相関値が急速に減衰し、他の2つの設定よりもはるかに早くゼロに近づきます。この挙動は、回路の深さとゲート数が大きいことと一致しており、蓄積されたノイズが長距離の相関を急速に劣化させます。

`pm_2`と`pm_3`はどちらも、基本的にすべての距離にわたって`pm_1`に対して大幅な改善を示しています。平均的には、`pm_3`が最も強い全体的な性能を示し、より長い距離にわたってより高い相関値を維持し、より緩やかな減衰を示しています。これは、より積極的な最適化と一致しており、ノイズの蓄積に対して一般的により堅牢な浅い回路を生成します。

とはいえ、`pm_2`は`pm_3`と比較して、わずかに大きな深さとゲート数を持つにもかかわらず、短距離では明らかに優れた精度を示しています。このことは、回路の深さだけでは性能を完全に決定できないことを示唆しています。エンタングルゲートの配置やエラーが回路を通じてどのように伝播するかなど、トランスパイルによって生成される具体的な構造も重要な役割を果たします。場合によっては、`pm_2`によって適用される変換が、長距離ではスケールしないものの、局所的な相関をより良く保持しているように見えます。

これらの結果を総合すると、回路のコンパクトさと回路の構造の間のトレードオフが浮き彫りになります。最適化の強化は一般的に長距離の安定性を向上させますが、特定のオブザーバブルに対する最良の性能は、回路の深さの削減とハードウェアのノイズ特性に適合した構造の生成の両方に依存します。
## パートII. SABREにおけるヒューリスティックの設定とServerlessの使用
トライアル数の調整に加えて、SABREはトランスパイル時に使用されるルーティングヒューリスティックのカスタマイズにも対応しています。デフォルトでは、`SabreLayout` はdecayヒューリスティックを使用しており、スワップされる可能性に基づいて量子ビットを動的に重み付けします。別のヒューリスティック（例えば `lookahead` ヒューリスティック）を使用するには、カスタム `SabreSwap` パスを作成し、`FullAncillaAllocation`、`EnlargeWithAncilla`、および `ApplyLayout` を含む `PassManager` を実行して `SabreLayout` に接続します。`SabreSwap` を `SabreLayout` のパラメータとして使用する場合、デフォルトではレイアウトトライアルは1回のみ実行されます。複数のレイアウトトライアルを効率的に実行するために、サーバーレスランタイムを活用して並列化を行います。サーバーレスの詳細については、[Serverlessドキュメント](/guides/serverless)を参照してください。

### ルーティングヒューリスティックの変更方法
1. 目的のヒューリスティックを使用してカスタム `SabreSwap` パスを作成します。
2. このカスタム `SabreSwap` を `SabreLayout` パスのルーティング手法として使用します。

ループを使用して複数のレイアウトトライアルを実行することも可能ですが、大規模でより本格的な実験にはサーバーレスランタイムの方が適しています。サーバーレスはレイアウトトライアルの並列実行をサポートしており、大規模な回路や大規模な実験スイープの最適化を大幅に高速化します。これは、リソース集約型のタスクや時間効率が重要な場合に特に有用です。

このセクションでは、最適化のステップ2、すなわち回路のサイズと深さを最小化して最良のトランスパイル済み回路を得ることに焦点を当てます。前述の結果を踏まえ、ヒューリスティックのカスタマイズとサーバーレスの並列化が最適化性能をさらに向上させ、大規模な量子回路トランスパイルに適したものにする方法を探ります。
### サーバーレスランタイムなしの結果（レイアウトトライアル1回）

In [29]:
job_lookahead = transpile_remote_serverless.run(
    circuit=qc,
    backend_name=backend.name,
    optimization_level=3,
    seed_list=seed_list,
    swap_trials=swap_trials,
    heuristic="lookahead",
)

In [30]:
job_lookahead.job_id

'15767dfc-e71d-4720-94d6-9212f72334c2'

In [31]:
job_lookahead.status()

'QUEUED'

Receive the logs and results from the serverless runtime.

In [21]:
logs_lookahead = job_lookahead.logs()
print(logs_lookahead)

No logs yet.


Once a program is `DONE`, you can use `job.results()` to fetch the result stored in `save_result()`.

In [32]:
# Run the job with lookahead heuristic
start_time = time.time()
results_lookahead = job_lookahead.result()
end_time = time.time()

job_lookahead_time = end_time - start_time

以下のセルでは、`transpile_remote.py` ファイルを `transpile_remote_serverless` という名前でQiskit Serverlessプログラムとしてアップロードします。

In [33]:
job_decay = transpile_remote_serverless.run(
    circuit=qc,
    backend_name=backend.name,
    optimization_level=3,
    seed_list=seed_list,
    swap_trials=swap_trials,
    heuristic="decay",
)

In [34]:
job_decay.job_id

'00418c76-d6ec-4bd8-9f70-05d0fa14d4eb'

In [35]:
logs_decay = job_decay.logs()
print(logs_decay)

No logs yet.


In [36]:
# Run the job with the decay heuristic
start_time = time.time()
results_decay = job_decay.result()
end_time = time.time()

job_decay_time = end_time - start_time

In [37]:
# Extract transpilation times
transpile_times_decay = results_decay["transpile_times"]
transpile_times_lookahead = results_lookahead["transpile_times"]

# Calculate total transpilation time for serial execution
total_transpile_time_decay = sum(transpile_times_decay)
total_transpile_time_lookahead = sum(transpile_times_lookahead)

# Print total transpilation time
print("=== Total Transpilation Time (Serial Execution) ===")
print(f"Decay Heuristic    : {total_transpile_time_decay:.2f} seconds")
print(f"Lookahead Heuristic: {total_transpile_time_lookahead:.2f} seconds")

# Print serverless job time (parallel execution)
print("\n=== Serverless Job Time (Parallel Execution) ===")
print(f"Decay Heuristic    : {job_decay_time:.2f} seconds")
print(f"Lookahead Heuristic: {job_lookahead_time:.2f} seconds")

# Calculate and print average runtime per transpilation
avg_transpile_time_decay = total_transpile_time_decay / num_seeds
avg_transpile_time_lookahead = total_transpile_time_lookahead / num_seeds
avg_job_time_decay = job_decay_time / num_seeds
avg_job_time_lookahead = job_lookahead_time / num_seeds

print("\n=== Average Time Per Transpilation ===")
print(f"Decay Heuristic (Serial)    : {avg_transpile_time_decay:.2f} seconds")
print(f"Decay Heuristic (Serverless): {avg_job_time_decay:.2f} seconds")
print(
    f"Lookahead Heuristic (Serial)    : {avg_transpile_time_lookahead:.2f} seconds"
)
print(
    f"Lookahead Heuristic (Serverless): {avg_job_time_lookahead:.2f} seconds"
)

# Calculate and print serverless improvement percentage
decay_improvement_percentage = (
    (total_transpile_time_decay - job_decay_time) / total_transpile_time_decay
) * 100
lookahead_improvement_percentage = (
    (total_transpile_time_lookahead - job_lookahead_time)
    / total_transpile_time_lookahead
) * 100

print("\n=== Serverless Improvement ===")
print(f"Decay Heuristic    : {decay_improvement_percentage:.2f}%")
print(f"Lookahead Heuristic: {lookahead_improvement_percentage:.2f}%")

=== Total Transpilation Time (Serial Execution) ===
Decay Heuristic    : 112.37 seconds
Lookahead Heuristic: 85.37 seconds

=== Serverless Job Time (Parallel Execution) ===
Decay Heuristic    : 5.72 seconds
Lookahead Heuristic: 5.85 seconds

=== Average Time Per Transpilation ===
Decay Heuristic (Serial)    : 5.62 seconds
Decay Heuristic (Serverless): 0.29 seconds
Lookahead Heuristic (Serial)    : 4.27 seconds
Lookahead Heuristic (Serverless): 0.29 seconds

=== Serverless Improvement ===
Decay Heuristic    : 94.91%
Lookahead Heuristic: 93.14%


These results demonstrate the substantial efficiency gains from using serverless execution for quantum circuit transpilation. Compared to serial execution, serverless execution dramatically reduces overall runtime for both the `decay` and `lookahead` heuristics by parallelizing independent transpilation trials. While serial execution reflects the full cumulative cost of exploring multiple layout trials, the serverless job times highlight how parallel execution collapses this cost into a much shorter wall-clock time. As a result, the effective time per transpilation is reduced to a small fraction of that required in the serial setting, largely independent of the heuristic used. This capability is particularly important for optimizing SABRE to its fullest potential. Many of SABRE’s strongest performance gains come from increasing the number of layout and routing trials, which can be prohibitively expensive when executed sequentially. Serverless execution removes this bottleneck, enabling large-scale parameter sweeps and deeper exploration of heuristic configurations with minimal overhead.

Overall, these findings show that serverless execution is key to scaling SABRE optimization, making aggressive experimentation and refinement practical compared to serial execution.

Obtain the results from the serverless runtime and compare the results of the lookahead and decay heuristics. We will compare the sizes and depths.

In [38]:
# Extract sizes and depths
sizes_lookahead = [
    circuit.size() for circuit in results_lookahead["transpiled_circuits"]
]
depths_lookahead = [
    circuit.depth(lambda x: x.operation.num_qubits == 2)
    for circuit in results_lookahead["transpiled_circuits"]
]
sizes_decay = [
    circuit.size() for circuit in results_decay["transpiled_circuits"]
]
depths_decay = [
    circuit.depth(lambda x: x.operation.num_qubits == 2)
    for circuit in results_decay["transpiled_circuits"]
]


def create_scatterplot(x, y1, y2, xlabel, ylabel, title, labels, colors):
    plt.figure(figsize=(8, 5))
    plt.scatter(
        x, y1, label=labels[0], color=colors[0], alpha=0.8, edgecolor="k"
    )
    plt.scatter(
        x, y2, label=labels[1], color=colors[1], alpha=0.8, edgecolor="k"
    )
    plt.xlabel(xlabel, fontsize=12)
    plt.ylabel(ylabel, fontsize=12)
    plt.title(title, fontsize=14)
    plt.legend(fontsize=10)
    plt.grid(axis="y", linestyle="--", alpha=0.7)
    plt.tight_layout()
    plt.show()


create_scatterplot(
    seed_list,
    sizes_lookahead,
    sizes_decay,
    "Seed",
    "Size",
    "Circuit Size",
    ["lookahead", "Decay"],
    ["blue", "red"],
)
create_scatterplot(
    seed_list,
    depths_lookahead,
    depths_decay,
    "Seed",
    "Depth",
    "Circuit Depth",
    ["lookahead", "Decay"],
    ["blue", "red"],
)

<Image src="../docs/images/tutorials/transpilation-optimizations-with-sabre/extracted-outputs/4cf9588b-8ea6-4761-b544-14bef8f0be85-0.avif" alt="Output of the previous code cell" />

<Image src="../docs/images/tutorials/transpilation-optimizations-with-sabre/extracted-outputs/4cf9588b-8ea6-4761-b544-14bef8f0be85-1.avif" alt="Output of the previous code cell" />

Each point in the scatter plots above represents a layout trial, with the x-axis indicating the circuit depth and the y-axis indicating the circuit size. The results reveal that the lookahead heuristic generally outperforms the decay heuristic in minimizing circuit depth and circuit size. In practical applications, the goal is to identify the optimal layout trial for your chosen heuristic, whether prioritizing depth or size. This can be achieved by selecting the trial with the lowest value for the desired metric. Importantly, increasing the number of layout trials improves the chances of achieving a better result in terms of size or depth, but it comes at the cost of higher computational overhead.

In [39]:
min_depth_lookahead = min(depths_lookahead)
min_depth_decay = min(depths_decay)
min_size_lookahead = min(sizes_lookahead)
min_size_decay = min(sizes_decay)
print(
    "Lookahead: Min Depth",
    min_depth_lookahead,
    "Min Size",
    min_size_lookahead,
)
print("Decay:     Min Depth", min_depth_decay, "Min Size", min_size_decay)

Lookahead: Min Depth 399 Min Size 2452
Decay:     Min Depth 415 Min Size 2611


サーバーレスランタイムからログと結果を受け取ります。

In [40]:
# This cell is hidden from users, it cleans up the `source_files` directory
from pathlib import Path

Path("source_files/transpile_remote.py").unlink()
Path("source_files").rmdir()

## Conclusion

In this tutorial, we explored how to optimize large circuits using SABRE in Qiskit. We demonstrated how to configure the `SabreLayout` pass with different parameters to balance circuit quality and transpilation runtime. We also showed how to customize the routing heuristic in SABRE and use the `QiskitServerless`runtime to parallelize layout trials efficiently for when `SabreSwap` is involved. By adjusting these parameters and heuristics, you can optimize the layout and routing of large circuits, ensuring they are executed efficiently on quantum hardware.

## Tutorial survey

Please take this short survey to provide feedback on this tutorial. Your insights will help us improve our content offerings and user experience.

[Link to survey](https://your.feedback.ibm.com/jfe/form/SV_d9YWUSQIAvU9HXE)